# DEH 30-Day PySpark Challenge
## Day 22 — Practice Set 1: Filtering & Aggregations

**Instructions:**
1. Click `File → Save a copy in Drive` before editing
2. For each problem: write SQL first, then PySpark, in the empty cells
3. Reference solutions follow each problem — try first before checking

---

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

spark = (
    SparkSession.builder
    .appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

In [ ]:
orders_schema = StructType([
    StructField("order_id",       StringType(),  True),
    StructField("customer_id",    StringType(),  True),
    StructField("product_id",     StringType(),  True),
    StructField("order_date",     DateType(),    True),
    StructField("quantity",       IntegerType(), True),
    StructField("unit_price",     DoubleType(),  True),
    StructField("discount_pct",   IntegerType(), True),
    StructField("status",         StringType(),  True),
    StructField("payment_method", StringType(),  True),
    StructField("region",         StringType(),  True)
])

orders_df = spark.read.option("header","true").schema(orders_schema).csv(f"s3a://{BUCKET}/data/orders.csv")
orders_df.createOrReplaceTempView("orders")
print(f"Orders: {orders_df.count()}")

---
## Problem 1 (Easy) — Orders above a threshold, sorted

Find all orders with `unit_price` > 300 AND `status` = 'Delivered'.  
Return `order_id`, `customer_id`, `unit_price`, sorted by `unit_price` descending.

In [ ]:
# Your SQL attempt


In [ ]:
# Your PySpark attempt


### Reference Solution — Problem 1

In [ ]:
# SQL solution
spark.sql("""
    SELECT order_id, customer_id, unit_price
    FROM orders
    WHERE unit_price > 300 AND status = 'Delivered'
    ORDER BY unit_price DESC
""").show(5)

In [ ]:
# PySpark solution
orders_df.filter(
    (F.col("unit_price") > 300) & (F.col("status") == "Delivered")
).select("order_id", "customer_id", "unit_price") \
 .orderBy(F.col("unit_price").desc()) \
 .show(5)

---
## Problem 2 (Easy) — Revenue and order count by region

For each `region`, calculate total revenue (`unit_price * quantity`) and order count. Sort by total revenue descending.

In [ ]:
# Your SQL attempt


In [ ]:
# Your PySpark attempt


### Reference Solution — Problem 2

In [ ]:
spark.sql("""
    SELECT
        region,
        ROUND(SUM(unit_price * quantity), 2) AS total_revenue,
        COUNT(order_id) AS order_count
    FROM orders
    GROUP BY region
    ORDER BY total_revenue DESC
""").show()

In [ ]:
orders_df.groupBy("region").agg(
    F.round(F.sum(F.col("unit_price") * F.col("quantity")), 2).alias("total_revenue"),
    F.count("order_id").alias("order_count")
).orderBy(F.col("total_revenue").desc()).show()

---
## Problem 3 (Medium) — Regions with more than N orders

Find regions with more than 20 orders. Show `region` and `order_count`.

In [ ]:
# Your SQL attempt


In [ ]:
# Your PySpark attempt


### Reference Solution — Problem 3

In [ ]:
spark.sql("""
    SELECT region, COUNT(order_id) AS order_count
    FROM orders
    GROUP BY region
    HAVING COUNT(order_id) > 20
    ORDER BY order_count DESC
""").show()

In [ ]:
orders_df.groupBy("region").agg(
    F.count("order_id").alias("order_count")
).filter(F.col("order_count") > 20) \
 .orderBy(F.col("order_count").desc()) \
 .show()

---
## Problem 4 (Medium) — Average order value by payment method, excluding cancelled

Excluding `status = 'Cancelled'`, calculate average `unit_price` per `payment_method`. Round to 2 decimals. Sort descending.

In [ ]:
# Your SQL attempt


In [ ]:
# Your PySpark attempt


### Reference Solution — Problem 4

In [ ]:
spark.sql("""
    SELECT
        payment_method,
        ROUND(AVG(unit_price), 2) AS avg_order_value
    FROM orders
    WHERE status != 'Cancelled'
    GROUP BY payment_method
    ORDER BY avg_order_value DESC
""").show()

In [ ]:
orders_df.filter(F.col("status") != "Cancelled") \
    .groupBy("payment_method").agg(
        F.round(F.avg("unit_price"), 2).alias("avg_order_value")
    ).orderBy(F.col("avg_order_value").desc()).show()

---
## Problem 5 (Hard) — Customers with above-average spend

Find customers whose total spend is greater than the overall average total spend per customer.

In [ ]:
# Your SQL attempt


In [ ]:
# Your PySpark attempt


### Reference Solution — Problem 5

In [ ]:
spark.sql("""
    WITH customer_spend AS (
        SELECT customer_id, SUM(unit_price) AS total_spend
        FROM orders
        GROUP BY customer_id
    )
    SELECT customer_id, total_spend
    FROM customer_spend
    WHERE total_spend > (SELECT AVG(total_spend) FROM customer_spend)
    ORDER BY total_spend DESC
""").show(10)

In [ ]:
customer_spend = orders_df.groupBy("customer_id").agg(
    F.sum("unit_price").alias("total_spend")
)

# Get the overall average as a scalar value
avg_spend = customer_spend.agg(F.avg("total_spend")).collect()[0][0]

customer_spend.filter(F.col("total_spend") > avg_spend) \
    .orderBy(F.col("total_spend").desc()) \
    .show(10)

---
## Problem 6 (Hard) — Multi-level grouping with conditional aggregation

Per `region`: total order count, count of Delivered orders, and % delivered (1 decimal).

In [ ]:
# Your SQL attempt


In [ ]:
# Your PySpark attempt


### Reference Solution — Problem 6

In [ ]:
spark.sql("""
    SELECT
        region,
        COUNT(order_id) AS total_orders,
        SUM(CASE WHEN status = 'Delivered' THEN 1 ELSE 0 END) AS delivered_orders,
        ROUND(
            100.0 * SUM(CASE WHEN status = 'Delivered' THEN 1 ELSE 0 END) / COUNT(order_id),
            1
        ) AS pct_delivered
    FROM orders
    GROUP BY region
    ORDER BY pct_delivered DESC
""").show()

In [ ]:
orders_df.groupBy("region").agg(
    F.count("order_id").alias("total_orders"),
    F.sum(F.when(F.col("status") == "Delivered", 1).otherwise(0)).alias("delivered_orders")
).withColumn(
    "pct_delivered",
    F.round(100.0 * F.col("delivered_orders") / F.col("total_orders"), 1)
).orderBy(F.col("pct_delivered").desc()).show()

---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*